# 07 — Latest Prediction Snapshot

This notebook loads the saved MarketGuard models and generates one latest prediction row per stock.

Outputs:

- Probability of outperforming Nifty 50 over the next 20 trading days
- Probability of a 10% downside event during the next 20 trading days
- Downside-risk band
- Prediction readiness and data-quality information

## Load data, models, and metadata

In [1]:
from pathlib import Path
import json

import joblib
import pandas as pd


def find_project_root(start_path: Path) -> Path:
    """Find the MarketGuard repository root."""
    start_path = start_path.resolve()

    for path in [start_path, *start_path.parents]:
        if (path / "config.yaml").exists() or (path / "README.md").exists():
            return path

    raise FileNotFoundError(
        "Could not find the project root containing config.yaml or README.md."
    )


def extract_feature_names(model) -> list[str]:
    """Extract the exact training feature order from a saved sklearn pipeline."""

    if hasattr(model, "feature_names_in_"):
        return list(model.feature_names_in_)

    if hasattr(model, "named_steps"):
        for step in model.named_steps.values():
            if hasattr(step, "feature_names_in_"):
                return list(step.feature_names_in_)

    raise AttributeError(
        "Feature names were not stored in the saved model pipeline."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

TARGET_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "targets"
    / "stock_features_with_targets_v1.parquet"
)

MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports" / "baseline_modeling"

SNAPSHOT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "predictions"
)

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)


OUTPERFORM_MODEL_PATH = (
    MODEL_DIR
    / "best_random_forest_outperform_nifty50_20d_v1.joblib"
)

DOWNSIDE_MODEL_PATH = (
    MODEL_DIR
    / "random_forest_downside_10pct_20d_v1.joblib"
)

OUTPERFORM_METADATA_PATH = (
    REPORT_DIR
    / "outperform_nifty50_20d_model_metadata_v1.json"
)

DOWNSIDE_METADATA_PATH = (
    REPORT_DIR
    / "downside_10pct_model_metadata_v1.json"
)


required_paths = [
    TARGET_DATA_PATH,
    OUTPERFORM_MODEL_PATH,
    DOWNSIDE_MODEL_PATH,
    OUTPERFORM_METADATA_PATH,
    DOWNSIDE_METADATA_PATH,
]

missing_paths = [
    path for path in required_paths
    if not path.exists()
]

if missing_paths:
    missing_text = "\n".join(str(path) for path in missing_paths)

    raise FileNotFoundError(
        f"Required inference assets are missing:\n{missing_text}"
    )


# Load full feature/target dataset
data = pd.read_parquet(TARGET_DATA_PATH)

data["date"] = pd.to_datetime(data["date"])

data = (
    data
    .sort_values(["yf_ticker", "date"])
    .reset_index(drop=True)
)


# Load saved trained models
outperform_model = joblib.load(OUTPERFORM_MODEL_PATH)
downside_model = joblib.load(DOWNSIDE_MODEL_PATH)


# Extract exact feature order from each model
outperform_feature_cols = extract_feature_names(outperform_model)
downside_feature_cols = extract_feature_names(downside_model)

if outperform_feature_cols != downside_feature_cols:
    raise ValueError(
        "The outperform and downside models use different feature orders."
    )

feature_cols = outperform_feature_cols


# Load metadata
with open(
    OUTPERFORM_METADATA_PATH,
    "r",
    encoding="utf-8",
) as file:
    outperform_metadata = json.load(file)

with open(
    DOWNSIDE_METADATA_PATH,
    "r",
    encoding="utf-8",
) as file:
    downside_metadata = json.load(file)


# Confirm all required model features exist
missing_feature_cols = [
    column
    for column in feature_cols
    if column not in data.columns
]

if missing_feature_cols:
    raise ValueError(
        f"Current dataset is missing model features: {missing_feature_cols}"
    )


print("Project root:")
print(PROJECT_ROOT)

print("\nDataset shape:")
print(data.shape)

print("\nDataset date range:")
print(data["date"].min(), "to", data["date"].max())

print("\nUnique stocks:")
print(data["yf_ticker"].nunique())

print("\nLoaded models:")
print("Outperform model:", type(outperform_model))
print("Downside model:", type(downside_model))

print("\nFeature count:")
print(len(feature_cols))

print("\nBoth models use identical feature order:")
print(outperform_feature_cols == downside_feature_cols)

print("\nSnapshot output directory:")
print(SNAPSHOT_DIR)

Project root:
E:\Projects\marketguard-india

Dataset shape:
(357370, 200)

Dataset date range:
2010-01-04 00:00:00 to 2026-07-14 00:00:00

Unique stocks:
100

Loaded models:
Outperform model: <class 'sklearn.pipeline.Pipeline'>
Downside model: <class 'sklearn.pipeline.Pipeline'>

Feature count:
79

Both models use identical feature order:
True

Snapshot output directory:
E:\Projects\marketguard-india\data\processed\predictions


### Select latest stock rows and check readiness

In [2]:
import numpy as np

# Select the latest available feature row for every stock
latest_rows = (
    data
    .sort_values(["yf_ticker", "date"])
    .groupby("yf_ticker", group_keys=False)
    .tail(1)
    .copy()
    .sort_values("yf_ticker")
    .reset_index(drop=True)
)

# Check model-feature completeness
latest_feature_data = (
    latest_rows[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
)

latest_rows["missing_model_feature_count"] = (
    latest_feature_data
    .isna()
    .sum(axis=1)
)

latest_rows["all_model_features_available"] = (
    latest_rows["missing_model_feature_count"] == 0
)

# model_ready_v1 contains the historical eligibility rules used during modeling
if "model_ready_v1" in latest_rows.columns:
    latest_rows["prediction_ready"] = (
        latest_rows["all_model_features_available"]
        & latest_rows["model_ready_v1"].astype(bool)
    )
else:
    latest_rows["prediction_ready"] = (
        latest_rows["all_model_features_available"]
    )

latest_date_summary = (
    latest_rows["date"]
    .value_counts()
    .sort_index(ascending=False)
    .rename_axis("latest_date")
    .reset_index(name="stock_count")
)

readiness_summary = (
    latest_rows["prediction_ready"]
    .value_counts(dropna=False)
    .rename_axis("prediction_ready")
    .reset_index(name="stock_count")
)

print("Latest rows shape:")
print(latest_rows.shape)

print("\nUnique stocks:")
print(latest_rows["yf_ticker"].nunique())

print("\nLatest date distribution:")
display(latest_date_summary)

print("\nPrediction readiness:")
display(readiness_summary)

print("\nMissing model-feature count distribution:")
display(
    latest_rows["missing_model_feature_count"]
    .value_counts()
    .sort_index()
    .rename_axis("missing_feature_count")
    .reset_index(name="stock_count")
)

# Show stocks that are not ready
not_ready_cols = [
    col
    for col in [
        "date",
        "symbol",
        "company_name",
        "industry",
        "yf_ticker",
        "stock_acceptance",
        "history_days",
        "model_ready_v1",
        "missing_model_feature_count",
        "prediction_ready",
    ]
    if col in latest_rows.columns
]

not_ready_stocks = latest_rows.loc[
    ~latest_rows["prediction_ready"],
    not_ready_cols,
].copy()

print("\nStocks not ready for normal-confidence prediction:")
display(not_ready_stocks)

Latest rows shape:
(100, 203)

Unique stocks:
100

Latest date distribution:


,latest_date,stock_count
0,2026-07-14,100



Prediction readiness:


,prediction_ready,stock_count
0,True,90
1,False,10



Missing model-feature count distribution:


,missing_feature_count,stock_count
0,0,98
1,2,2



Stocks not ready for normal-confidence prediction:


,date,symbol,company_name,industry,yf_ticker,model_ready_v1,missing_model_feature_count,prediction_ready
31,2026-07-14,ENRIN,Siemens Energy India Ltd.,Capital Goods,ENRIN.NS,0,0,False
32,2026-07-14,ETERNAL,Eternal Ltd.,Consumer Services,ETERNAL.NS,0,0,False
44,2026-07-14,HYUNDAI,Hyundai Motor India Ltd.,Automobile and Auto Components,HYUNDAI.NS,0,0,False
50,2026-07-14,IRFC,Indian Railway Finance Corporation Ltd.,Financial Services,IRFC.NS,0,0,False
53,2026-07-14,JIOFIN,Jio Financial Services Ltd.,Financial Services,JIOFIN.NS,0,0,False
56,2026-07-14,LODHA,Lodha Developers Ltd.,Realty,LODHA.NS,0,0,False
61,2026-07-14,MAXHEALTH,Max Healthcare Institute Ltd.,Healthcare,MAXHEALTH.NS,0,0,False
62,2026-07-14,MAZDOCK,Mazagoan Dock Shipbuilders Ltd.,Capital Goods,MAZDOCK.NS,0,0,False
81,2026-07-14,TATACAP,Tata Capital Ltd.,Financial Services,TATACAP.NS,0,2,False
88,2026-07-14,TMCV,Tata Motors Ltd.,Capital Goods,TMCV.NS,0,2,False


-90 stocks are fully ready under the same eligibility rules used during training.

-8 stocks have all 79 model features but insufficient historical data.

-TATACAP and TMCV have limited history plus two missing features.

-Both saved pipelines contain a median imputer, so predictions can still be generated for all 100 stocks, but the 10 short-history stocks must carry a lower-confidence flag.

### Generate latest predictions for all stocks

In [3]:
# Generate predictions for the latest row of all 100 stocks

X_latest = (
    latest_rows[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
)

# Raw model probability outputs
latest_rows["outperform_probability"] = (
    outperform_model.predict_proba(X_latest)[:, 1]
)

latest_rows["downside_probability"] = (
    downside_model.predict_proba(X_latest)[:, 1]
)


def assign_downside_risk_band(probability: float) -> str:
    """
    Convert downside model probability into the risk bands
    selected during threshold analysis.
    """
    if probability < 0.40:
        return "Low Risk"

    if probability < 0.47:
        return "Watch Risk"

    if probability < 0.51:
        return "High Risk"

    return "Very High Risk"


latest_rows["downside_risk_band"] = (
    latest_rows["downside_probability"]
    .apply(assign_downside_risk_band)
)


# Confidence/readiness label
conditions = [
    latest_rows["prediction_ready"],
    (
        ~latest_rows["prediction_ready"]
        & (latest_rows["missing_model_feature_count"] == 0)
    ),
    latest_rows["missing_model_feature_count"] > 0,
]

confidence_labels = [
    "Normal Confidence",
    "Limited History",
    "Limited History + Missing Features",
]

latest_rows["prediction_confidence"] = np.select(
    conditions,
    confidence_labels,
    default="Review Required",
)


# Rank only the 90 research-ready stocks
latest_rows["outperform_rank_ready_universe"] = pd.Series(
    pd.NA,
    index=latest_rows.index,
    dtype="Int64",
)

latest_rows["downside_risk_rank_ready_universe"] = pd.Series(
    pd.NA,
    index=latest_rows.index,
    dtype="Int64",
)

ready_mask = latest_rows["prediction_ready"]

latest_rows.loc[
    ready_mask,
    "outperform_rank_ready_universe",
] = (
    latest_rows.loc[ready_mask, "outperform_probability"]
    .rank(method="first", ascending=False)
    .astype("Int64")
)

latest_rows.loc[
    ready_mask,
    "downside_risk_rank_ready_universe",
] = (
    latest_rows.loc[ready_mask, "downside_probability"]
    .rank(method="first", ascending=False)
    .astype("Int64")
)


# Final snapshot columns
snapshot_columns = [
    column
    for column in [
        "date",
        "symbol",
        "company_name",
        "industry",
        "yf_ticker",
        "outperform_probability",
        "outperform_rank_ready_universe",
        "downside_probability",
        "downside_risk_rank_ready_universe",
        "downside_risk_band",
        "prediction_ready",
        "prediction_confidence",
        "missing_model_feature_count",
        "model_ready_v1",
    ]
    if column in latest_rows.columns
]

latest_prediction_snapshot = (
    latest_rows[snapshot_columns]
    .sort_values(
        [
            "prediction_ready",
            "downside_probability",
        ],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)


print("Snapshot rows:")
print(len(latest_prediction_snapshot))

print("\nPrediction confidence summary:")
display(
    latest_prediction_snapshot["prediction_confidence"]
    .value_counts()
    .rename_axis("prediction_confidence")
    .reset_index(name="stock_count")
)

print("\nDownside risk-band summary:")
display(
    latest_prediction_snapshot["downside_risk_band"]
    .value_counts()
    .rename_axis("downside_risk_band")
    .reset_index(name="stock_count")
)

print("\nTop 15 downside-risk stocks from the ready universe:")
display(
    latest_prediction_snapshot.loc[
        latest_prediction_snapshot["prediction_ready"]
    ]
    .sort_values("downside_probability", ascending=False)
    .head(15)
)

print("\nTop 15 outperform-ranked stocks from the ready universe:")
display(
    latest_prediction_snapshot.loc[
        latest_prediction_snapshot["prediction_ready"]
    ]
    .sort_values("outperform_probability", ascending=False)
    .head(15)
)

print("\nLimited-confidence stocks:")
display(
    latest_prediction_snapshot.loc[
        ~latest_prediction_snapshot["prediction_ready"]
    ]
)

Snapshot rows:
100

Prediction confidence summary:


,prediction_confidence,stock_count
0,Normal Confidence,90
1,Limited History,8
2,Limited History + Missing Features,2



Downside risk-band summary:


,downside_risk_band,stock_count
0,Very High Risk,41
1,Watch Risk,29
2,Low Risk,21
3,High Risk,9



Top 15 downside-risk stocks from the ready universe:


,date,symbol,company_name,industry,yf_ticker,outperform_probability,outperform_rank_ready_universe,downside_probability,downside_risk_rank_ready_universe,downside_risk_band,prediction_ready,prediction_confidence,missing_model_feature_count,model_ready_v1
0,2026-07-14,VEDL,Vedanta Ltd.,Metals & Mining,VEDL.NS,0.428796,90,0.709686,1,Very High Risk,True,Normal Confidence,0,1
1,2026-07-14,HCLTECH,HCL Technologies Ltd.,Information Technology,HCLTECH.NS,0.489580,40,0.644713,2,Very High Risk,True,Normal Confidence,0,1
2,2026-07-14,LTM,LTM Ltd.,Information Technology,LTM.NS,0.480629,54,0.644368,3,Very High Risk,True,Normal Confidence,0,1
3,2026-07-14,INFY,Infosys Ltd.,Information Technology,INFY.NS,0.526448,5,0.632757,4,Very High Risk,True,Normal Confidence,0,1
4,2026-07-14,TMPV,Tata Motors Passenger Vehicles Ltd.,Automobile and Auto Components,TMPV.NS,0.444622,82,0.621657,5,Very High Risk,True,Normal Confidence,0,1
5,2026-07-14,MUTHOOTFIN,Muthoot Finance Ltd.,Financial Services,MUTHOOTFIN.NS,0.449065,81,0.609122,6,Very High Risk,True,Normal Confidence,0,1
6,2026-07-14,TCS,Tata Consultancy Services Ltd.,Information Technology,TCS.NS,0.498468,23,0.604039,7,Very High Risk,True,Normal Confidence,0,1
7,2026-07-14,DLF,DLF Ltd.,Realty,DLF.NS,0.468422,77,0.592496,8,Very High Risk,True,Normal Confidence,0,1
8,2026-07-14,TRENT,Trent Ltd.,Consumer Services,TRENT.NS,0.472336,72,0.585936,9,Very High Risk,True,Normal Confidence,0,1
9,2026-07-14,HINDZINC,Hindustan Zinc Ltd.,Metals & Mining,HINDZINC.NS,0.477906,58,0.565261,10,Very High Risk,True,Normal Confidence,0,1



Top 15 outperform-ranked stocks from the ready universe:


,date,symbol,company_name,industry,yf_ticker,outperform_probability,outperform_rank_ready_universe,downside_probability,downside_risk_rank_ready_universe,downside_risk_band,prediction_ready,prediction_confidence,missing_model_feature_count,model_ready_v1
88,2026-07-14,APOLLOHOSP,Apollo Hospitals Enterprise Ltd.,Healthcare,APOLLOHOSP.NS,0.564380,1,0.330996,89,Low Risk,True,Normal Confidence,0,1
85,2026-07-14,GRASIM,Grasim Industries Ltd.,Construction Materials,GRASIM.NS,0.537724,2,0.346938,86,Low Risk,True,Normal Confidence,0,1
84,2026-07-14,PIDILITIND,Pidilite Industries Ltd.,Chemicals,PIDILITIND.NS,0.536066,3,0.355195,85,Low Risk,True,Normal Confidence,0,1
26,2026-07-14,WIPRO,Wipro Ltd.,Information Technology,WIPRO.NS,0.527187,4,0.527350,27,Very High Risk,True,Normal Confidence,0,1
3,2026-07-14,INFY,Infosys Ltd.,Information Technology,INFY.NS,0.526448,5,0.632757,4,Very High Risk,True,Normal Confidence,0,1
83,2026-07-14,LT,Larsen & Toubro Ltd.,Construction,LT.NS,0.518347,6,0.357374,84,Low Risk,True,Normal Confidence,0,1
74,2026-07-14,TITAN,Titan Company Ltd.,Consumer Durables,TITAN.NS,0.513995,7,0.386657,75,Low Risk,True,Normal Confidence,0,1
76,2026-07-14,ASIANPAINT,Asian Paints Ltd.,Consumer Durables,ASIANPAINT.NS,0.512050,8,0.380597,77,Low Risk,True,Normal Confidence,0,1
81,2026-07-14,ULTRACEMCO,UltraTech Cement Ltd.,Construction Materials,ULTRACEMCO.NS,0.510594,9,0.364067,82,Low Risk,True,Normal Confidence,0,1
77,2026-07-14,SUNPHARMA,Sun Pharmaceutical Industries Ltd.,Healthcare,SUNPHARMA.NS,0.509656,10,0.376481,78,Low Risk,True,Normal Confidence,0,1



Limited-confidence stocks:


,date,symbol,company_name,industry,yf_ticker,outperform_probability,outperform_rank_ready_universe,downside_probability,downside_risk_rank_ready_universe,downside_risk_band,prediction_ready,prediction_confidence,missing_model_feature_count,model_ready_v1
90,2026-07-14,LODHA,Lodha Developers Ltd.,Realty,LODHA.NS,0.453679,<NA>,0.633697,<NA>,Very High Risk,False,Limited History,0,0
91,2026-07-14,ENRIN,Siemens Energy India Ltd.,Capital Goods,ENRIN.NS,0.508897,<NA>,0.588763,<NA>,Very High Risk,False,Limited History,0,0
92,2026-07-14,IRFC,Indian Railway Finance Corporation Ltd.,Financial Services,IRFC.NS,0.437978,<NA>,0.555464,<NA>,Very High Risk,False,Limited History,0,0
93,2026-07-14,JIOFIN,Jio Financial Services Ltd.,Financial Services,JIOFIN.NS,0.432662,<NA>,0.551387,<NA>,Very High Risk,False,Limited History,0,0
94,2026-07-14,TMCV,Tata Motors Ltd.,Capital Goods,TMCV.NS,0.485238,<NA>,0.539134,<NA>,Very High Risk,False,Limited History + Missing Features,2,0
95,2026-07-14,ETERNAL,Eternal Ltd.,Consumer Services,ETERNAL.NS,0.460599,<NA>,0.536487,<NA>,Very High Risk,False,Limited History,0,0
96,2026-07-14,HYUNDAI,Hyundai Motor India Ltd.,Automobile and Auto Components,HYUNDAI.NS,0.436781,<NA>,0.534577,<NA>,Very High Risk,False,Limited History,0,0
97,2026-07-14,MAZDOCK,Mazagoan Dock Shipbuilders Ltd.,Capital Goods,MAZDOCK.NS,0.450854,<NA>,0.510898,<NA>,Very High Risk,False,Limited History,0,0
98,2026-07-14,TATACAP,Tata Capital Ltd.,Financial Services,TATACAP.NS,0.479352,<NA>,0.509903,<NA>,High Risk,False,Limited History + Missing Features,2,0
99,2026-07-14,MAXHEALTH,Max Healthcare Institute Ltd.,Healthcare,MAXHEALTH.NS,0.486263,<NA>,0.491806,<NA>,High Risk,False,Limited History,0,0


The pipeline is working correctly. A few important interpretations:

APOLLOHOSP, GRASIM, and PIDILITIND currently rank highly for outperformance while remaining in the low-risk band.

WIPRO and INFY rank highly for outperformance but also have very high downside scores. This is not contradictory: they may have upside potential while also carrying significant drawdown risk.

41 Very High Risk does not mean 41 stocks will definitely fall 10%. It means their model score exceeded the 0.51 alert threshold. These probabilities are mainly useful for ranking and risk bands, not as perfectly calibrated real-world probabilities.

The 10 limited-history stocks should remain in the output but should not be ranked with the 90 normal-confidence stocks.

Next, save the snapshot so the dashboard and future notebooks can load it directly.

### Save latest prediction snapshot

In [4]:
from datetime import datetime
import json

snapshot_date = latest_prediction_snapshot["date"].max()
snapshot_date_text = snapshot_date.strftime("%Y-%m-%d")

FULL_SNAPSHOT_CSV_PATH = (
    SNAPSHOT_DIR
    / f"latest_prediction_snapshot_{snapshot_date_text}_v1.csv"
)

FULL_SNAPSHOT_PARQUET_PATH = (
    SNAPSHOT_DIR
    / f"latest_prediction_snapshot_{snapshot_date_text}_v1.parquet"
)

READY_SNAPSHOT_CSV_PATH = (
    SNAPSHOT_DIR
    / f"latest_prediction_snapshot_ready_{snapshot_date_text}_v1.csv"
)

# Stable filenames that can be used later by the dashboard
LATEST_SNAPSHOT_CSV_PATH = (
    SNAPSHOT_DIR / "latest_prediction_snapshot_v1.csv"
)

LATEST_SNAPSHOT_PARQUET_PATH = (
    SNAPSHOT_DIR / "latest_prediction_snapshot_v1.parquet"
)

SNAPSHOT_METADATA_PATH = (
    SNAPSHOT_DIR / "latest_prediction_snapshot_metadata_v1.json"
)


# Full snapshot: all 100 stocks
latest_prediction_snapshot.to_csv(
    FULL_SNAPSHOT_CSV_PATH,
    index=False,
)

latest_prediction_snapshot.to_parquet(
    FULL_SNAPSHOT_PARQUET_PATH,
    index=False,
)


# Ready-universe snapshot: 90 normal-confidence stocks
ready_prediction_snapshot = (
    latest_prediction_snapshot.loc[
        latest_prediction_snapshot["prediction_ready"]
    ]
    .copy()
    .reset_index(drop=True)
)

ready_prediction_snapshot.to_csv(
    READY_SNAPSHOT_CSV_PATH,
    index=False,
)


# Stable latest files for dashboard use
latest_prediction_snapshot.to_csv(
    LATEST_SNAPSHOT_CSV_PATH,
    index=False,
)

latest_prediction_snapshot.to_parquet(
    LATEST_SNAPSHOT_PARQUET_PATH,
    index=False,
)


# Snapshot metadata
snapshot_metadata = {
    "snapshot_version": "v1",
    "prediction_date": snapshot_date_text,
    "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "total_stocks": int(len(latest_prediction_snapshot)),
    "normal_confidence_stocks": int(
        latest_prediction_snapshot["prediction_ready"].sum()
    ),
    "limited_confidence_stocks": int(
        (~latest_prediction_snapshot["prediction_ready"]).sum()
    ),
    "feature_count": int(len(feature_cols)),
    "outperform_model": outperform_metadata.get(
        "model_name",
        "best_random_forest",
    ),
    "downside_model": downside_metadata.get(
        "model_name",
        "random_forest_downside_balanced",
    ),
    "risk_band_counts_all_stocks": {
        str(key): int(value)
        for key, value in (
            latest_prediction_snapshot["downside_risk_band"]
            .value_counts()
            .to_dict()
            .items()
        )
    },
    "notes": [
        "Probabilities are model-estimated scores and are not yet calibrated.",
        "Only prediction_ready=True stocks are included in normal-confidence ranks.",
        "Limited-history stocks are retained but excluded from ready-universe rankings.",
    ],
}

with open(
    SNAPSHOT_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(snapshot_metadata, file, indent=4)


print("Saved full dated CSV:")
print(FULL_SNAPSHOT_CSV_PATH)

print("\nSaved full dated Parquet:")
print(FULL_SNAPSHOT_PARQUET_PATH)

print("\nSaved ready-universe CSV:")
print(READY_SNAPSHOT_CSV_PATH)

print("\nSaved stable latest CSV:")
print(LATEST_SNAPSHOT_CSV_PATH)

print("\nSaved stable latest Parquet:")
print(LATEST_SNAPSHOT_PARQUET_PATH)

print("\nSaved snapshot metadata:")
print(SNAPSHOT_METADATA_PATH)

print("\nRows saved:")
print("All stocks:", len(latest_prediction_snapshot))
print("Ready universe:", len(ready_prediction_snapshot))

Saved full dated CSV:
E:\Projects\marketguard-india\data\processed\predictions\latest_prediction_snapshot_2026-07-14_v1.csv

Saved full dated Parquet:
E:\Projects\marketguard-india\data\processed\predictions\latest_prediction_snapshot_2026-07-14_v1.parquet

Saved ready-universe CSV:
E:\Projects\marketguard-india\data\processed\predictions\latest_prediction_snapshot_ready_2026-07-14_v1.csv

Saved stable latest CSV:
E:\Projects\marketguard-india\data\processed\predictions\latest_prediction_snapshot_v1.csv

Saved stable latest Parquet:
E:\Projects\marketguard-india\data\processed\predictions\latest_prediction_snapshot_v1.parquet

Saved snapshot metadata:
E:\Projects\marketguard-india\data\processed\predictions\latest_prediction_snapshot_metadata_v1.json

Rows saved:
All stocks: 100
Ready universe: 90


Next, combine the two model outputs into an opportunity-versus-risk classification.

We should not combine the raw probabilities mathematically yet because the outperform model is mainly a ranking model and its probabilities are not calibrated. Instead, we will use:

Outperform rank among the 90 ready stocks.

Downside-risk band.

Prediction confidence.

### Create opportunity-versus-risk classification

In [5]:
import math

combined_snapshot = latest_prediction_snapshot.copy()

ready_count = int(combined_snapshot["prediction_ready"].sum())

# Opportunity groups based on rank among the ready universe
high_opportunity_rank_limit = math.ceil(ready_count * 0.20)
moderate_opportunity_rank_limit = math.ceil(ready_count * 0.50)


def assign_opportunity_tier(row) -> str:
    """
    Classify opportunity using the outperform rank.

    Top 20% = High Opportunity
    Next 30% = Moderate Opportunity
    Bottom 50% = Low Opportunity
    """
    if not row["prediction_ready"]:
        return "Limited Confidence"

    rank = row["outperform_rank_ready_universe"]

    if rank <= high_opportunity_rank_limit:
        return "High Opportunity"

    if rank <= moderate_opportunity_rank_limit:
        return "Moderate Opportunity"

    return "Low Opportunity"


combined_snapshot["opportunity_tier"] = combined_snapshot.apply(
    assign_opportunity_tier,
    axis=1,
)


def assign_opportunity_risk_class(row) -> str:
    """
    Combine outperform ranking and downside-risk band.
    """
    if not row["prediction_ready"]:
        return "Limited Confidence - Review"

    opportunity = row["opportunity_tier"]
    risk = row["downside_risk_band"]

    lower_risk = risk in ["Low Risk", "Watch Risk"]
    higher_risk = risk in ["High Risk", "Very High Risk"]

    if opportunity == "High Opportunity" and lower_risk:
        return "Attractive Risk-Reward"

    if opportunity == "High Opportunity" and higher_risk:
        return "High Opportunity / High Risk"

    if opportunity == "Moderate Opportunity" and lower_risk:
        return "Balanced Opportunity"

    if opportunity == "Moderate Opportunity" and higher_risk:
        return "Caution"

    if opportunity == "Low Opportunity" and lower_risk:
        return "Low Opportunity / Lower Risk"

    return "Unfavourable Risk-Reward"


combined_snapshot["opportunity_risk_class"] = combined_snapshot.apply(
    assign_opportunity_risk_class,
    axis=1,
)


print("Ready-universe size:")
print(ready_count)

print("\nOpportunity rank limits:")
print("High Opportunity: ranks 1 to", high_opportunity_rank_limit)
print(
    "Moderate Opportunity: ranks",
    high_opportunity_rank_limit + 1,
    "to",
    moderate_opportunity_rank_limit,
)
print(
    "Low Opportunity: ranks",
    moderate_opportunity_rank_limit + 1,
    "to",
    ready_count,
)

print("\nOpportunity-tier summary:")
display(
    combined_snapshot["opportunity_tier"]
    .value_counts()
    .rename_axis("opportunity_tier")
    .reset_index(name="stock_count")
)

print("\nCombined opportunity-risk summary:")
display(
    combined_snapshot["opportunity_risk_class"]
    .value_counts()
    .rename_axis("opportunity_risk_class")
    .reset_index(name="stock_count")
)

display_columns = [
    "date",
    "symbol",
    "company_name",
    "industry",
    "outperform_probability",
    "outperform_rank_ready_universe",
    "opportunity_tier",
    "downside_probability",
    "downside_risk_band",
    "opportunity_risk_class",
    "prediction_confidence",
]

print("\nAttractive risk-reward stocks:")
display(
    combined_snapshot.loc[
        combined_snapshot["opportunity_risk_class"]
        == "Attractive Risk-Reward",
        display_columns,
    ]
    .sort_values("outperform_rank_ready_universe")
)

print("\nHigh opportunity but high-risk stocks:")
display(
    combined_snapshot.loc[
        combined_snapshot["opportunity_risk_class"]
        == "High Opportunity / High Risk",
        display_columns,
    ]
    .sort_values("outperform_rank_ready_universe")
)

print("\nUnfavourable risk-reward stocks:")
display(
    combined_snapshot.loc[
        combined_snapshot["opportunity_risk_class"]
        == "Unfavourable Risk-Reward",
        display_columns,
    ]
    .sort_values("downside_probability", ascending=False)
    .head(20)
)

Ready-universe size:
90

Opportunity rank limits:
High Opportunity: ranks 1 to 18
Moderate Opportunity: ranks 19 to 45
Low Opportunity: ranks 46 to 90

Opportunity-tier summary:


,opportunity_tier,stock_count
0,Low Opportunity,45
1,Moderate Opportunity,27
2,High Opportunity,18
3,Limited Confidence,10



Combined opportunity-risk summary:


,opportunity_risk_class,stock_count
0,Unfavourable Risk-Reward,27
1,Balanced Opportunity,22
2,Low Opportunity / Lower Risk,18
3,Attractive Risk-Reward,10
4,Limited Confidence - Review,10
5,High Opportunity / High Risk,8
6,Caution,5



Attractive risk-reward stocks:


,date,symbol,company_name,industry,outperform_probability,outperform_rank_ready_universe,opportunity_tier,downside_probability,downside_risk_band,opportunity_risk_class,prediction_confidence
88,2026-07-14,APOLLOHOSP,Apollo Hospitals Enterprise Ltd.,Healthcare,0.564380,1,High Opportunity,0.330996,Low Risk,Attractive Risk-Reward,Normal Confidence
85,2026-07-14,GRASIM,Grasim Industries Ltd.,Construction Materials,0.537724,2,High Opportunity,0.346938,Low Risk,Attractive Risk-Reward,Normal Confidence
84,2026-07-14,PIDILITIND,Pidilite Industries Ltd.,Chemicals,0.536066,3,High Opportunity,0.355195,Low Risk,Attractive Risk-Reward,Normal Confidence
83,2026-07-14,LT,Larsen & Toubro Ltd.,Construction,0.518347,6,High Opportunity,0.357374,Low Risk,Attractive Risk-Reward,Normal Confidence
74,2026-07-14,TITAN,Titan Company Ltd.,Consumer Durables,0.513995,7,High Opportunity,0.386657,Low Risk,Attractive Risk-Reward,Normal Confidence
76,2026-07-14,ASIANPAINT,Asian Paints Ltd.,Consumer Durables,0.512050,8,High Opportunity,0.380597,Low Risk,Attractive Risk-Reward,Normal Confidence
81,2026-07-14,ULTRACEMCO,UltraTech Cement Ltd.,Construction Materials,0.510594,9,High Opportunity,0.364067,Low Risk,Attractive Risk-Reward,Normal Confidence
77,2026-07-14,SUNPHARMA,Sun Pharmaceutical Industries Ltd.,Healthcare,0.509656,10,High Opportunity,0.376481,Low Risk,Attractive Risk-Reward,Normal Confidence
72,2026-07-14,TORNTPHARM,Torrent Pharmaceuticals Ltd.,Healthcare,0.507233,11,High Opportunity,0.388495,Low Risk,Attractive Risk-Reward,Normal Confidence
60,2026-07-14,HAL,Hindustan Aeronautics Ltd.,Capital Goods,0.507181,12,High Opportunity,0.423639,Watch Risk,Attractive Risk-Reward,Normal Confidence



High opportunity but high-risk stocks:


,date,symbol,company_name,industry,outperform_probability,outperform_rank_ready_universe,opportunity_tier,downside_probability,downside_risk_band,opportunity_risk_class,prediction_confidence
26,2026-07-14,WIPRO,Wipro Ltd.,Information Technology,0.527187,4,High Opportunity,0.527350,Very High Risk,High Opportunity / High Risk,Normal Confidence
3,2026-07-14,INFY,Infosys Ltd.,Information Technology,0.526448,5,High Opportunity,0.632757,Very High Risk,High Opportunity / High Risk,Normal Confidence
24,2026-07-14,CGPOWER,CG Power and Industrial Solutions Ltd.,Capital Goods,0.505924,13,High Opportunity,0.527998,Very High Risk,High Opportunity / High Risk,Normal Confidence
36,2026-07-14,SIEMENS,Siemens Ltd.,Capital Goods,0.504853,14,High Opportunity,0.485967,High Risk,High Opportunity / High Risk,Normal Confidence
33,2026-07-14,ABB,ABB India Ltd.,Capital Goods,0.504668,15,High Opportunity,0.503660,High Risk,High Opportunity / High Risk,Normal Confidence
20,2026-07-14,MOTHERSON,Samvardhana Motherson International Ltd.,Automobile and Auto Components,0.504273,16,High Opportunity,0.536311,Very High Risk,High Opportunity / High Risk,Normal Confidence
27,2026-07-14,SOLARINDS,Solar Industries India Ltd.,Chemicals,0.503902,17,High Opportunity,0.527297,Very High Risk,High Opportunity / High Risk,Normal Confidence
35,2026-07-14,CUMMINSIND,Cummins India Ltd.,Capital Goods,0.503856,18,High Opportunity,0.494918,High Risk,High Opportunity / High Risk,Normal Confidence



Unfavourable risk-reward stocks:


,date,symbol,company_name,industry,outperform_probability,outperform_rank_ready_universe,opportunity_tier,downside_probability,downside_risk_band,opportunity_risk_class,prediction_confidence
0,2026-07-14,VEDL,Vedanta Ltd.,Metals & Mining,0.428796,90,Low Opportunity,0.709686,Very High Risk,Unfavourable Risk-Reward,Normal Confidence
2,2026-07-14,LTM,LTM Ltd.,Information Technology,0.480629,54,Low Opportunity,0.644368,Very High Risk,Unfavourable Risk-Reward,Normal Confidence
4,2026-07-14,TMPV,Tata Motors Passenger Vehicles Ltd.,Automobile and Auto Components,0.444622,82,Low Opportunity,0.621657,Very High Risk,Unfavourable Risk-Reward,Normal Confidence
5,2026-07-14,MUTHOOTFIN,Muthoot Finance Ltd.,Financial Services,0.449065,81,Low Opportunity,0.609122,Very High Risk,Unfavourable Risk-Reward,Normal Confidence
7,2026-07-14,DLF,DLF Ltd.,Realty,0.468422,77,Low Opportunity,0.592496,Very High Risk,Unfavourable Risk-Reward,Normal Confidence
8,2026-07-14,TRENT,Trent Ltd.,Consumer Services,0.472336,72,Low Opportunity,0.585936,Very High Risk,Unfavourable Risk-Reward,Normal Confidence
9,2026-07-14,HINDZINC,Hindustan Zinc Ltd.,Metals & Mining,0.477906,58,Low Opportunity,0.565261,Very High Risk,Unfavourable Risk-Reward,Normal Confidence
10,2026-07-14,ADANIENSOL,Adani Energy Solutions Ltd.,Power,0.472682,69,Low Opportunity,0.563798,Very High Risk,Unfavourable Risk-Reward,Normal Confidence
11,2026-07-14,ADANIGREEN,Adani Green Energy Ltd.,Power,0.469959,76,Low Opportunity,0.553072,Very High Risk,Unfavourable Risk-Reward,Normal Confidence
12,2026-07-14,BPCL,Bharat Petroleum Corporation Ltd.,Oil Gas & Consumable Fuels,0.441877,85,Low Opportunity,0.551110,Very High Risk,Unfavourable Risk-Reward,Normal Confidence


## Latest Prediction Snapshot — Summary

The snapshot contains predictions for **100 NIFTY 100 stocks** using data from **2026-07-14**.

- **90 stocks** have normal-confidence predictions.
- **8 stocks** have limited history.
- **2 stocks** have limited history and missing features.

### How to Read the Table

| Column | Meaning |
|---|---|
| `outperform_probability` | Model score for outperforming Nifty 50 over the next 20 trading days |
| `outperform_rank_ready_universe` | Opportunity rank among the 90 ready stocks; rank 1 is strongest |
| `opportunity_tier` | High, Moderate, or Low Opportunity |
| `downside_probability` | Model score for a possible 10% drawdown within 20 trading days |
| `downside_risk_band` | Low, Watch, High, or Very High Risk |
| `opportunity_risk_class` | Combined opportunity and downside-risk interpretation |
| `prediction_confidence` | Data quality and historical-data availability |

### Combined Classification

| Classification | Interpretation |
|---|---|
| Attractive Risk-Reward | High opportunity with lower downside risk |
| High Opportunity / High Risk | Strong opportunity signal but elevated downside risk |
| Balanced Opportunity | Moderate opportunity with lower risk |
| Caution | Moderate opportunity with elevated risk |
| Low Opportunity / Lower Risk | Lower opportunity but relatively lower risk |
| Unfavourable Risk-Reward | Low opportunity with elevated risk |
| Limited Confidence – Review | Insufficient history or missing features |

### Confidence

`prediction_confidence` represents **data confidence**, not certainty that the prediction is correct.

- **Normal Confidence:** sufficient history and complete features
- **Limited History:** complete features but insufficient history
- **Limited History + Missing Features:** short history and missing inputs

The outperform model should mainly be used for **ranking**, while the downside model is more useful for identifying relative drawdown risk. These outputs are decision-support signals, not investment recommendations.

### Save final combined snapshot
Next, save the combined opportunity-risk snapshot. We will also rename prediction_confidence to data_confidence so it is not mistaken for model certainty.

In [6]:
from datetime import datetime
import json

# Prepare final output
combined_snapshot_final = combined_snapshot.copy()

combined_snapshot_final["data_confidence"] = (
    combined_snapshot_final["prediction_confidence"]
)

combined_snapshot_final = combined_snapshot_final.drop(
    columns=["prediction_confidence"]
)

combined_snapshot_final = (
    combined_snapshot_final
    .sort_values(
        [
            "prediction_ready",
            "outperform_rank_ready_universe",
            "downside_probability",
        ],
        ascending=[False, True, True],
        na_position="last",
    )
    .reset_index(drop=True)
)

combined_ready_snapshot = (
    combined_snapshot_final.loc[
        combined_snapshot_final["prediction_ready"]
    ]
    .copy()
    .reset_index(drop=True)
)


snapshot_date = combined_snapshot_final["date"].max()
snapshot_date_text = snapshot_date.strftime("%Y-%m-%d")


# Dated files
DATED_COMBINED_CSV_PATH = (
    SNAPSHOT_DIR
    / f"combined_opportunity_risk_snapshot_{snapshot_date_text}_v1.csv"
)

DATED_COMBINED_PARQUET_PATH = (
    SNAPSHOT_DIR
    / f"combined_opportunity_risk_snapshot_{snapshot_date_text}_v1.parquet"
)

DATED_READY_CSV_PATH = (
    SNAPSHOT_DIR
    / f"combined_opportunity_risk_snapshot_ready_{snapshot_date_text}_v1.csv"
)


# Stable files for dashboard use
LATEST_COMBINED_CSV_PATH = (
    SNAPSHOT_DIR
    / "latest_combined_opportunity_risk_snapshot_v1.csv"
)

LATEST_COMBINED_PARQUET_PATH = (
    SNAPSHOT_DIR
    / "latest_combined_opportunity_risk_snapshot_v1.parquet"
)

COMBINED_METADATA_PATH = (
    SNAPSHOT_DIR
    / "latest_combined_opportunity_risk_metadata_v1.json"
)


# Save dated files
combined_snapshot_final.to_csv(
    DATED_COMBINED_CSV_PATH,
    index=False,
)

combined_snapshot_final.to_parquet(
    DATED_COMBINED_PARQUET_PATH,
    index=False,
)

combined_ready_snapshot.to_csv(
    DATED_READY_CSV_PATH,
    index=False,
)


# Save stable latest files
combined_snapshot_final.to_csv(
    LATEST_COMBINED_CSV_PATH,
    index=False,
)

combined_snapshot_final.to_parquet(
    LATEST_COMBINED_PARQUET_PATH,
    index=False,
)


# Save metadata
combined_metadata = {
    "snapshot_version": "v1",
    "prediction_date": snapshot_date_text,
    "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "total_stocks": int(len(combined_snapshot_final)),
    "ready_universe_stocks": int(len(combined_ready_snapshot)),
    "limited_confidence_stocks": int(
        (~combined_snapshot_final["prediction_ready"]).sum()
    ),
    "feature_count": int(len(feature_cols)),
    "confidence_field": {
        "name": "data_confidence",
        "meaning": (
            "Historical-data availability and feature completeness, "
            "not certainty that the model prediction is correct."
        ),
    },
    "opportunity_tiers": {
        "high": f"ranks 1 to {high_opportunity_rank_limit}",
        "moderate": (
            f"ranks {high_opportunity_rank_limit + 1} "
            f"to {moderate_opportunity_rank_limit}"
        ),
        "low": (
            f"ranks {moderate_opportunity_rank_limit + 1} "
            f"to {ready_count}"
        ),
    },
    "classification_counts": {
        str(label): int(count)
        for label, count in (
            combined_snapshot_final["opportunity_risk_class"]
            .value_counts()
            .to_dict()
            .items()
        )
    },
    "notes": [
        "Outperform scores are primarily intended for relative ranking.",
        "Downside scores are not yet probability-calibrated.",
        "Combined classes are decision-support categories.",
        "Limited-confidence stocks are excluded from ready-universe ranks.",
        "Historical backtesting is required before strategy use.",
    ],
}

with open(
    COMBINED_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(combined_metadata, file, indent=4)


print("Saved dated combined CSV:")
print(DATED_COMBINED_CSV_PATH)

print("\nSaved dated combined Parquet:")
print(DATED_COMBINED_PARQUET_PATH)

print("\nSaved ready-universe CSV:")
print(DATED_READY_CSV_PATH)

print("\nSaved stable dashboard CSV:")
print(LATEST_COMBINED_CSV_PATH)

print("\nSaved stable dashboard Parquet:")
print(LATEST_COMBINED_PARQUET_PATH)

print("\nSaved metadata:")
print(COMBINED_METADATA_PATH)

print("\nRows saved:")
print("All stocks:", len(combined_snapshot_final))
print("Ready universe:", len(combined_ready_snapshot))

Saved dated combined CSV:
E:\Projects\marketguard-india\data\processed\predictions\combined_opportunity_risk_snapshot_2026-07-14_v1.csv

Saved dated combined Parquet:
E:\Projects\marketguard-india\data\processed\predictions\combined_opportunity_risk_snapshot_2026-07-14_v1.parquet

Saved ready-universe CSV:
E:\Projects\marketguard-india\data\processed\predictions\combined_opportunity_risk_snapshot_ready_2026-07-14_v1.csv

Saved stable dashboard CSV:
E:\Projects\marketguard-india\data\processed\predictions\latest_combined_opportunity_risk_snapshot_v1.csv

Saved stable dashboard Parquet:
E:\Projects\marketguard-india\data\processed\predictions\latest_combined_opportunity_risk_snapshot_v1.parquet

Saved metadata:
E:\Projects\marketguard-india\data\processed\predictions\latest_combined_opportunity_risk_metadata_v1.json

Rows saved:
All stocks: 100
Ready universe: 90


### Validate saved output

In [7]:
# Reload the stable output to verify it is usable outside this notebook

validated_snapshot = pd.read_parquet(
    LATEST_COMBINED_PARQUET_PATH
)

required_output_columns = [
    "date",
    "symbol",
    "company_name",
    "industry",
    "outperform_probability",
    "outperform_rank_ready_universe",
    "downside_probability",
    "downside_risk_band",
    "opportunity_tier",
    "opportunity_risk_class",
    "prediction_ready",
    "data_confidence",
]

missing_output_columns = [
    column
    for column in required_output_columns
    if column not in validated_snapshot.columns
]

if missing_output_columns:
    raise ValueError(
        f"Saved snapshot is missing columns: {missing_output_columns}"
    )

assert len(validated_snapshot) == 100
assert validated_snapshot["symbol"].nunique() == 100
assert int(validated_snapshot["prediction_ready"].sum()) == 90
assert validated_snapshot["date"].nunique() == 1

print("Saved snapshot validation passed.")

print("\nShape:")
print(validated_snapshot.shape)

print("\nPrediction date:")
print(validated_snapshot["date"].max())

print("\nUnique stocks:")
print(validated_snapshot["symbol"].nunique())

print("\nReady stocks:")
print(int(validated_snapshot["prediction_ready"].sum()))

print("\nLimited-confidence stocks:")
print(int((~validated_snapshot["prediction_ready"]).sum()))

print("\nRequired output columns present:")
print(True)

Saved snapshot validation passed.

Shape:
(100, 16)

Prediction date:
2026-07-14 00:00:00

Unique stocks:
100

Ready stocks:
90

Limited-confidence stocks:
10

Required output columns present:
True
